# PennyLane: Diferenciación Automática de Circuitos Cuánticos

**Laboratorio 29 · Nivel avanzado**

PennyLane es un framework de aprendizaje automático cuántico que integra
diferenciación automática con circuitos cuánticos. A diferencia de Qiskit,
permite calcular gradientes exactos de circuitos con respecto a sus parámetros.

## Contenido

1. Instalación y primeros pasos con PennyLane
2. Parameter-shift rule: gradiente exacto de circuitos cuánticos
3. Comparativa: parameter-shift vs. diferencias finitas
4. Optimización de VQE con gradiente automático
5. Barren plateaus: visualización del gradiente vs. profundidad
6. Comparativa de velocidad: PennyLane vs. Qiskit

In [ ]:
# Verificar instalación de PennyLane
try:
    import pennylane as qml
    print(f'PennyLane {qml.__version__} disponible')
except ImportError:
    print('PennyLane no instalado. Instalar con:')
    print('  pip install pennylane')
    raise

import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import SparsePauliOp

## 1. Primeros pasos con PennyLane

En PennyLane un **dispositivo** (`device`) define el backend (simulador o hardware).
Un **QNode** es la unidad básica: un circuito cuántico diferenciable.

In [ ]:
import pennylane as qml
import numpy as np

# Dispositivo simulador de statevector
dev = qml.device('default.qubit', wires=2)

@qml.qnode(dev)
def circuito_bell():
    qml.Hadamard(wires=0)
    qml.CNOT(wires=[0, 1])
    return qml.probs(wires=[0, 1])

probs = circuito_bell()
print('Probabilidades estado Bell:', probs)
print('P(00) =', probs[0], '  P(11) =', probs[3])

In [ ]:
# Circuito parametrizado: Ry(theta) en qubit 0
dev1 = qml.device('default.qubit', wires=1)

@qml.qnode(dev1)
def rotacion_y(theta):
    qml.RY(theta, wires=0)
    return qml.expval(qml.PauliZ(0))

thetas = np.linspace(0, 2*np.pi, 100)
exp_z  = [rotacion_y(t) for t in thetas]

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(thetas, exp_z, 'b-', lw=2, label='⟨Z⟩ = cos(θ)')
ax.plot(thetas, np.cos(thetas), 'r--', lw=1, alpha=0.6, label='cos(θ) teórico')
ax.set_xlabel('θ'); ax.set_ylabel('⟨Z⟩'); ax.set_title('Ry(θ)|0⟩: valor esperado de Z')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 2. Parameter-Shift Rule

La **parameter-shift rule** calcula el gradiente exacto de un circuito cuántico:

$$\frac{\partial}{\partial\theta} \langle O \rangle(\theta) = \frac{1}{2}\left[\langle O\rangle\left(\theta + \frac{\pi}{2}\right) - \langle O\rangle\left(\theta - \frac{\pi}{2}\right)\right]$$

Esto requiere exactamente **2 evaluaciones del circuito** por parámetro,
independientemente de la profundidad. Es compatible con hardware real.

In [ ]:
# Gradiente automático con PennyLane (usa parameter-shift por defecto)
grad_fn = qml.grad(rotacion_y)

theta_test = 0.7
grad_pennylane = grad_fn(theta_test)
grad_analitico = -np.sin(theta_test)  # d/dθ cos(θ) = -sin(θ)

# Implementación manual de parameter-shift
def param_shift_manual(f, theta, shift=np.pi/2):
    return (f(theta + shift) - f(theta - shift)) / 2

grad_manual = param_shift_manual(rotacion_y, theta_test)

print(f'θ = {theta_test:.3f}')
print(f'Gradiente PennyLane (auto):  {grad_pennylane:.8f}')
print(f'Gradiente manual (shift):    {grad_manual:.8f}')
print(f'Gradiente analítico:         {grad_analitico:.8f}')
print(f'Error PL vs analítico:       {abs(grad_pennylane - grad_analitico):.2e}')

In [ ]:
# Comparativa: parameter-shift vs. diferencias finitas (central)
def finite_diff(f, theta, eps=1e-5):
    return (f(theta + eps) - f(theta - eps)) / (2 * eps)

theta_range = np.linspace(0, 2*np.pi, 50)
grad_ps = np.array([param_shift_manual(rotacion_y, t) for t in theta_range])
grad_fd = np.array([finite_diff(rotacion_y, t) for t in theta_range])
grad_exact = -np.sin(theta_range)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(theta_range, grad_exact, 'k-', lw=2, label='Exacto')
ax1.plot(theta_range, grad_ps, 'b--', lw=1.5, label='Parameter-shift')
ax1.plot(theta_range, grad_fd, 'r:', lw=1.5, label='Dif. finitas (ε=1e-5)')
ax1.set_xlabel('θ'); ax1.set_ylabel('dE/dθ')
ax1.set_title('Gradiente: comparativa de métodos')
ax1.legend(); ax1.grid(alpha=0.3)

error_ps = np.abs(grad_ps - grad_exact)
error_fd = np.abs(grad_fd - grad_exact)
ax2.semilogy(theta_range, error_ps + 1e-16, 'b-', label=f'PS: max={error_ps.max():.1e}')
ax2.semilogy(theta_range, error_fd + 1e-16, 'r-', label=f'FD: max={error_fd.max():.1e}')
ax2.set_xlabel('θ'); ax2.set_ylabel('|error|')
ax2.set_title('Error absoluto vs. gradiente exacto')
ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 3. VQE con gradiente automático

Optimizamos el Hamiltoniano de H₂ usando `qml.GradientDescentOptimizer`
(descenso de gradiente exacto con parameter-shift).

In [ ]:
# Hamiltoniano H2 como operador PennyLane
coeffs = [-1.0523732, 0.39793742, -0.39793742, -0.01128010, 0.18093119, 0.18093119]
obs    = [
    qml.Identity(0) @ qml.Identity(1),
    qml.PauliZ(0)   @ qml.Identity(1),
    qml.Identity(0) @ qml.PauliZ(1),
    qml.PauliZ(0)   @ qml.PauliZ(1),
    qml.PauliX(0)   @ qml.PauliX(1),
    qml.PauliY(0)   @ qml.PauliY(1),
]
H2 = qml.Hamiltonian(coeffs, obs)

dev2 = qml.device('default.qubit', wires=2)

@qml.qnode(dev2, diff_method='parameter-shift')
def vqe_ansatz(params):
    # Ansatz: rotaciones + entrelazamiento
    qml.RY(params[0], wires=0)
    qml.RY(params[1], wires=1)
    qml.CNOT(wires=[0, 1])
    qml.RY(params[2], wires=0)
    qml.RY(params[3], wires=1)
    return qml.expval(H2)

# Optimización con gradiente exacto
opt    = qml.GradientDescentOptimizer(stepsize=0.2)
params = np.array([0.1, 0.2, 0.3, 0.4], requires_grad=True)

energias = []
for step in range(100):
    params, energy = opt.step_and_cost(vqe_ansatz, params)
    energias.append(float(energy))

E_exact = -1.13727
print(f'Energía VQE final:  {energias[-1]:.6f} Hartree')
print(f'Energía exacta FCI: {E_exact:.6f} Hartree')
print(f'Error:              {abs(energias[-1] - E_exact)*1000:.2f} mHartree')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(energias, 'b-', lw=2, label='VQE (parameter-shift)')
ax.axhline(E_exact, color='red', linestyle='--', lw=1.5, label=f'FCI = {E_exact:.5f} Ha')
ax.set_xlabel('Iteración'); ax.set_ylabel('Energía (Hartree)')
ax.set_title('Convergencia de VQE para H₂ con gradiente exacto (PennyLane)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print(f'Convergencia en {next(i for i,e in enumerate(energias) if abs(e-E_exact)<0.01)} iteraciones (error < 10 mHa)')

## 4. Barren Plateaus: gradiente vs. profundidad

Los barren plateaus son regiones del paisaje de optimización donde el gradiente
es exponencialmente pequeño. Ocurren en ansatze aleatorios profundos.

In [ ]:
def variance_gradiente(n_qubits: int, depth: int, n_samples: int = 50) -> float:
    """Estima la varianza del gradiente para un ansatz aleatorio."""
    dev_bp = qml.device('default.qubit', wires=n_qubits)

    def make_circuit(params):
        idx = 0
        for _ in range(depth):
            for q in range(n_qubits):
                qml.RY(params[idx], wires=q); idx += 1
                qml.RZ(params[idx], wires=q); idx += 1
            for q in range(n_qubits - 1):
                qml.CNOT(wires=[q, q+1])
        return qml.expval(qml.PauliZ(0))

    circuit = qml.QNode(make_circuit, dev_bp, diff_method='parameter-shift')
    n_params = 2 * n_qubits * depth
    gradients = []

    for _ in range(n_samples):
        params = np.random.uniform(-np.pi, np.pi, n_params, requires_grad=True)
        grad = qml.grad(circuit)(params)
        gradients.append(float(grad[0]))  # gradiente del primer parámetro

    return float(np.var(gradients))

# Sweep sobre profundidades para n=4 qubits
n_qubits_bp = 4
depths = [1, 2, 3, 4, 5]
print(f'Calculando varianza del gradiente para n={n_qubits_bp} qubits...')
variances = []
for d in depths:
    v = variance_gradiente(n_qubits_bp, d, n_samples=30)
    variances.append(v)
    print(f'  depth={d}: Var[grad] = {v:.2e}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.semilogy(depths, variances, 'o-', color='#8e44ad', lw=2, ms=8)
ax1.set_xlabel('Profundidad del ansatz'); ax1.set_ylabel('Var[∂E/∂θ₀]')
ax1.set_title(f'Barren Plateau: varianza del gradiente (n={n_qubits_bp})')
ax1.grid(alpha=0.3)

# Comparativa entre métodos de diferenciación
metodos = ['parameter-shift', 'backprop', 'finite-diff']
tiempos_relativos = [1.0, 0.3, 1.8]  # relativos (PS = 1.0)
precision = ['Exacta', 'Exacta (GPU)', 'Aprox. (ε)']
colores = ['#3498db', '#2ecc71', '#e74c3c']

bars = ax2.bar(metodos, tiempos_relativos, color=colores, alpha=0.85)
for bar, prec in zip(bars, precision):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             prec, ha='center', va='bottom', fontsize=9)
ax2.set_ylabel('Tiempo relativo (PS = 1.0)')
ax2.set_title('Comparativa de métodos de diferenciación')
ax2.grid(alpha=0.3, axis='y')

plt.suptitle('Diferenciación automática de circuitos con PennyLane', fontsize=12)
plt.tight_layout(); plt.show()

## 5. Comparativa: PennyLane vs. Qiskit

| Característica | PennyLane | Qiskit |
|---|---|---|
| Gradientes exactos | ✅ parameter-shift, backprop | ⚠️ Solo diferencias finitas (SPSAEstimator) |
| Integración con PyTorch/JAX | ✅ Nativa | ❌ Parcial |
| Hardware IBM real | ⚠️ Via plugin | ✅ Native |
| Optimizadores ML | ✅ Adam, RMSProp, etc. | ⚠️ COBYLA, SPSA |
| Velocidad simulación | ⚠️ Similar | ✅ Aer GPU |
| Ecosistema QEC | ❌ No | ✅ Completo |

In [ ]:
# Interoperabilidad: convertir circuito PennyLane → Qiskit
try:
    import pennylane as qml

    @qml.qnode(qml.device('default.qubit', wires=2))
    def circuito_pl(params):
        qml.RY(params[0], wires=0)
        qml.RY(params[1], wires=1)
        qml.CNOT(wires=[0, 1])
        return qml.state()

    # Extraer el circuito Qiskit equivalente (si está disponible el plugin)
    print('Circuito PennyLane → Qiskit:')
    tape = qml.workflow.construct_tape(circuito_pl)([0.5, 0.8])
    print(f'  Operaciones: {len(tape.operations)}')
    print(f'  Parámetros:  {tape.num_params}')
    for op in tape.operations:
        print(f'    {op.name}({op.parameters}) en wires {op.wires}')
except Exception as e:
    print(f'Nota: {e}')
    print('La interoperabilidad completa requiere pennylane-qiskit plugin.')

print('\nResumen:')
print('  PennyLane es ideal para investigación en QML y optimización con gradientes exactos.')
print('  Qiskit es superior para acceso a hardware IBM, QEC y circuitos de gran escala.')
print('  Ambos se complementan en un flujo de trabajo real.')